<a href="https://colab.research.google.com/github/Isabela-Tellez/BootcampIA/blob/main/04.%20Abril-30/Reto%20CRUD%20(Create%20%C2%B7%20Read%20%C2%B7%20Update%20%C2%B7%20Delete).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #4B2A82 0%, #1A1F4E 100%); padding: 40px; border-radius: 12px; color: white; text-align: center;">

<h1 style="color: white; margin: 0; font-size: 2.5em;">🛠️ CRUD sobre la tabla empleados</h1>
<h2 style="color: #2EC4B6; margin: 10px 0 20px 0; font-weight: 300;">Reto CRUD </h2>
<p style="font-size: 1.1em; margin: 0;">Create · Read · Update · Delete</p>


</div>

<div style="background: #F5F0FA; border-left: 6px solid #4B2A82; padding: 20px; border-radius: 6px; margin: 20px 0;">

| Bloque |  Contenido |
|---|---|
| Setup + exploración |  Conectar a la BD y verificar los datos |
| Reto 1 — CREATE | Función `crear_empleado()` |
| Reto 2 — READ | Funciones `buscar_empleado()` y `listar_empleados()` |
| Reto 3 — UPDATE | Función `actualizar_empleado()` |
| Reto 4 — DELETE | Función `borrar_empleado()` |

**Reglas:**
- ❌ No hay soluciones — solo **hints**.
- ✅ Podéis consultar las celdas del notebook anterior (el de modelado) y la documentación de `sqlite3`.
</div>

<div style="background: #E8F8F6; border-left: 6px solid #2EC4B6; padding: 20px; border-radius: 6px; margin: 20px 0;">

### 📐 Modelo E/R construido

```
┌─────────────────┐         ┌─────────────────┐         ┌─────────────────┐
│  DEPARTAMENTO   │         │    EMPLEADOS    │         │    PROYECTO     │
├─────────────────┤   1:N   ├─────────────────┤         ├─────────────────┤
│ 🔑 id           │─────────│ 🔑 id          │         │ 🔑 id          │
│   nombre        │         │   nombre        │         │   nombre        │
│   ubicacion     │         │   email         │         │   presupuesto   │
└─────────────────┘         │   salario       │         └─────────────────┘
                            │🔗departamento_id│                  │
                            └─────────────────┘                  │
                                     │                           │
                                     │  Empleado N:M Proyecto    │
                                     └─────────┬─────────────────┘
                                               │
                                    ┌──────────────────────┐
                                    │      ASIGNACIÓN      │
                                    │     (intermedia)     │
                                    ├──────────────────────┤
                                    │ 🔗 empleado_id  (PK) │
                                    │ 🔗 proyecto_id  (PK) │
                                    └──────────────────────┘
```

**Dos relaciones clave:**

| Relación | Tipo | Implementación |
|---|---|---|
| Departamento → Empleado | **1:N** | FK `departamento_id` en `empleados` |
| Empleado ↔ Proyecto | **N:M** | Tabla intermedia `asignaciones` |

</div>

<div style="background: #E8F8F6; border-left: 6px solid #2EC4B6; padding: 20px; border-radius: 6px; margin: 20px 0;">

### 📐 Recordatorio — Tabla `empleados`

```
┌────────────────────────────────────────────────────────────┐
│                         empleados                          │
├──────────────────┬─────────┬───────────────────────────────┤
│ Columna          │ Tipo    │ Restricciones                 │
├──────────────────┼─────────┼───────────────────────────────┤
│ id               │ INTEGER │ PRIMARY KEY · AUTOINCREMENT   │
│ nombre           │ TEXT    │ NOT NULL                      │
│ email            │ TEXT    │ NOT NULL · UNIQUE             │
│ salario          │ REAL    │ NOT NULL · CHECK (> 0)        │
│ departamento_id  │ INTEGER │ NOT NULL · FK→departamentos   │
└──────────────────┴─────────┴───────────────────────────────┘
```

**Departamentos disponibles** (para las FKs):

| id | nombre | ubicacion |
|---|---|---|
| 1 | Datos | Madrid |
| 2 | Frontend | Barcelona |
| 3 | Backend | Madrid |

**Restricciones que la BD va a validar:**
- `email` no se puede repetir (UNIQUE)
- `salario` tiene que ser mayor que 0 (CHECK)
- `departamento_id` tiene que existir en la tabla `departamentos` (FK)
- `nombre`, `email`, `salario` y `departamento_id` son obligatorios (NOT NULL)

</div>

---

## ⚙️ Setup — Conectar a la BD

Con el fichero creado ayer `technova.db`. Subidlo a archivos:

1. Panel izquierdo 📁 → botón **"Subir"** (icono de hoja con flecha ↑)
2. Seleccionad el fichero `technova.db`
3. Esperad a que aparezca en la lista de archivos
4. Ejecutad la celda siguiente


In [163]:
# ========================================
# SETUP — Ejecutad esta celda tal cual
# ========================================

import sqlite3
import pandas as pd

# Conectar a la BD que habéis subido
conn = sqlite3.connect('technova.db')

# Activar foreign keys (CRÍTICO en SQLite)
conn.execute('PRAGMA foreign_keys = ON')

# Función auxiliar para ver resultados como tabla
def sql(query, params=()):
    return pd.read_sql_query(query, conn, params=params)

print('✅ Conectado a technova.db')
print('✅ Foreign keys activadas')
print('✅ Función sql() lista')

✅ Conectado a technova.db
✅ Foreign keys activadas
✅ Función sql() lista


---

## 👀 Exploración rápida — ¿Qué hay en la BD?

Antes de escribir CRUD, verificad que los datos están ahí. Ejecutad las siguientes celdas para ver el estado actual.

In [164]:
# Ver los departamentos disponibles (los necesitaréis para los INSERT)
sql("SELECT * FROM departamentos")

,id,nombre,ubicacion
0,1,Datos,Madrid
1,2,Frontend,Barcelona
2,3,Backend,Madrid


In [165]:
# Ver todos los empleados actuales
sql("SELECT * FROM empleados")

,id,nombre,email,salario,departamento_id
0,1,Ana García,ana@technova.com,45000.0,1
1,2,Luis Pérez,luis@technova.com,52000.0,1
2,3,María López,maria@technova.com,48000.0,2
3,4,Pablo Ruiz,pablo@technova.com,55000.0,3
4,5,Sara Díaz,sara@technova.com,60000.0,3


In [166]:
# Ver los proyectos disponibles
sql("SELECT * FROM proyectos")

,id,nombre,presupuesto
0,1,Chatbot Cliente,50000.0
1,2,Dashboard Analytics,30000.0
2,3,App Móvil,80000.0


In [167]:
# Ver las relaciones disponibles (Tabla intermedia)
sql("SELECT * FROM asignaciones")

,empleado_id,proyecto_id
0,1,1
1,1,2
2,2,1
3,3,3
4,4,1
5,4,3
6,5,2
7,5,3


In [168]:
# Ver cuántos empleados hay en total
sql("SELECT COUNT(*) AS total_empleados FROM empleados")

,total_empleados
0,5


---

<div style="background: #F5F0FA; border-left: 6px solid #4B2A82; padding: 20px; border-radius: 6px; margin: 20px 0;">

### 🎯 Reto 1 — CREATE


Cread una función `crear_empleado(nombre, email, salario, departamento_id)` que:

1. Inserte un nuevo empleado en la tabla `empleados`.
2. Devuelva el `id` del empleado creado.
3. Haga `commit()` para que el cambio se guarde.

**Después de escribir la función, probadla:**
- ✅ Caso correcto: `crear_empleado('Carlos Vega', 'carlos@technova.com', 42000, 2)`
- ❌ Caso con email duplicado: intentad crear otro con `'ana@technova.com'` (ya existe)
- ❌ Caso con salario negativo: intentad con salario `-1000`
- ❌ Caso con departamento inexistente: intentad con `departamento_id=99`

</div>

<div style="background: #FFF8E7; border-left: 6px solid #E8A317; padding: 15px; border-radius: 6px; margin: 10px 0;">

**💡 Hints:**
- El SQL necesario es: `INSERT INTO empleados (nombre, email, salario, departamento_id) VALUES (?, ?, ?, ?)`
- Los `?` son **placeholders** — se rellenan con una tupla de valores.
- `cursor.lastrowid` te da el id del registro recién creado.
- Para los casos de error, envolved el código en `try/except sqlite3.IntegrityError`.
- - No olvidéis el `conn.commit()` después del INSERT.

</div>

In [169]:
# ===========================
# RETO 1: Escribid aquí vuestra función crear_empleado()
# ===========================

def crear_empleado(nombre, email, salario, departamento_id):
    """
    Inserta un nuevo empleado en la BD.
    Devuelve el id creado si tiene éxito o o un mensaje de error si falla.
    """
    cursor = conn.execute("""
                  INSERT INTO empleados (nombre, email, salario, departamento_id)
                  VALUES (?, ?, ?, ?)
                  """, (nombre, email, salario, departamento_id))

    conn.commit()
    return cursor.lastrowid

In [170]:
# ===========================
# Probad vuestra función aquí
# ===========================

# ✅ Caso correcto
resultado = crear_empleado('Carlos Vega', 'carlos@technova.com', 42000, 2)
print(f'Empleado creado con id={resultado}')


# ❌ Probad los casos de error (descomentad uno a uno)
# print(crear_empleado('Otra Ana', 'ana@technova.com', 35000, 1))
# print(crear_empleado('Test', 'test@technova.com', -1000, 1))
# print(crear_empleado('Test', 'test2@technova.com', 30000, 99))



# Verificar que se ha insertado
sql("SELECT * FROM empleados")

Empleado creado con id=6


,id,nombre,email,salario,departamento_id
0,1,Ana García,ana@technova.com,45000.0,1
1,2,Luis Pérez,luis@technova.com,52000.0,1
2,3,María López,maria@technova.com,48000.0,2
3,4,Pablo Ruiz,pablo@technova.com,55000.0,3
4,5,Sara Díaz,sara@technova.com,60000.0,3
5,6,Carlos Vega,carlos@technova.com,42000.0,2


---

<div style="background: #F5F0FA; border-left: 6px solid #4B2A82; padding: 20px; border-radius: 6px; margin: 20px 0;">

### 🎯 Reto 2

Cread **dos funciones de lectura**:

**Función A — `buscar_empleado(id)`**
- Busca un empleado por su `id`.
- Si existe, devuelve un **diccionario** con sus datos: `{'id': 1, 'nombre': 'Ana García', ...}`
- Si no existe, devuelve `None`.

**Función B — `listar_empleados()`**
- Devuelve **todos** los empleados como una **lista de diccionarios**.
- Incluye el **nombre del departamento** (no solo el id numérico) usando un `JOIN`.

**Probadlas con:**

```python
# Empleado que existe
buscar_empleado(1)

# Empleado que NO existe
buscar_empleado(999)

# Listar todos con nombre de departamento
listar_empleados()
```

</div>

<div style="background: #FFF8E7; border-left: 6px solid #E8A317; padding: 15px; border-radius: 6px; margin: 10px 0;">

**💡 Hints:**
- `conn.execute(sql, params).fetchone()` devuelve **una** fila (o `None`).
- `conn.execute(sql, params).fetchall()` devuelve **todas** las filas.
- Para convertir filas en diccionarios: usad `conn.row_factory = sqlite3.Row` y luego `dict(fila)`.
- El JOIN para incluir el departamento:
  ```sql
  SELECT e.*, d.nombre AS departamento
  FROM empleados e
  INNER JOIN departamentos d ON e.departamento_id = d.id
  ```

</div>

In [171]:
# ===========================
# RETO 2A: Escribid aquí buscar_empleado()
# ===========================

def buscar_empleado(id):
    """
    Busca un empleado por id.
    Devuelve un diccionario con sus datos o None si no existe.
    """
    fila = conn.execute("SELECT id, nombre, email, salario, departamento_id FROM empleados WHERE id = ?", (id,)).fetchone()
    # Si no existe, se devuelve None
    if fila is None:
        return None
    # Se crea el diccionario manualmente
    return {
        'id': fila[0],
        'nombre': fila[1],
        'email': fila[2],
        'salario': fila[3],
        'departamento_id': fila[4]
    }

In [172]:
# ===========================
# RETO 2B: Escribid aquí listar_empleados()
# ===========================

def listar_empleados():
    """
    Devuelve todos los empleados como lista de diccionarios.
    Incluye el nombre del departamento (JOIN).
    """
    query = """
        SELECT e.*, d.nombre as nombre_departamento
        FROM empleados e
        JOIN departamentos d ON e.departamento_id = d.id
    """
    cursor = conn.execute(query)
    filas = cursor.fetchall()
    # Se convierte cada fila manualmente
    lista_resultado = []
    for fila in filas:
        empleado = {
            'id': fila[0],
            'nombre': fila[1],
            'email': fila[2],
            'salario': fila[3],
            'departamento_nombre': fila[4]
        }
        lista_resultado.append(empleado)

    return lista_resultado

In [173]:
# ===========================
# Probad vuestras funciones aquí
# ===========================

# Buscar un empleado que existe
print(buscar_empleado(1))

# Buscar un empleado que NO existe
print(buscar_empleado(999))

# Listar todos
for emp in listar_empleados():
  print(emp)

{'id': 1, 'nombre': 'Ana García', 'email': 'ana@technova.com', 'salario': 45000.0, 'departamento_id': 1}
None
{'id': 1, 'nombre': 'Ana García', 'email': 'ana@technova.com', 'salario': 45000.0, 'departamento_nombre': 1}
{'id': 2, 'nombre': 'Luis Pérez', 'email': 'luis@technova.com', 'salario': 52000.0, 'departamento_nombre': 1}
{'id': 3, 'nombre': 'María López', 'email': 'maria@technova.com', 'salario': 48000.0, 'departamento_nombre': 2}
{'id': 4, 'nombre': 'Pablo Ruiz', 'email': 'pablo@technova.com', 'salario': 55000.0, 'departamento_nombre': 3}
{'id': 5, 'nombre': 'Sara Díaz', 'email': 'sara@technova.com', 'salario': 60000.0, 'departamento_nombre': 3}
{'id': 6, 'nombre': 'Carlos Vega', 'email': 'carlos@technova.com', 'salario': 42000.0, 'departamento_nombre': 2}


---

<div style="background: #F5F0FA; border-left: 6px solid #4B2A82; padding: 20px; border-radius: 6px; margin: 20px 0;">

### 🎯 Reto 3 — UPDATE

Cread una función `actualizar_empleado(id, **kwargs)` que:

1. Reciba el `id` del empleado a actualizar.
2. Reciba **uno o varios campos** a cambiar como keyword arguments: `nombre=`, `email=`, `salario=`, `departamento_id=`.
3. Construya dinámicamente la consulta `UPDATE` con solo los campos proporcionados.
4. Devuelva `True` si actualizó alguna fila, `False` si el id no existía.
5. Capture posibles errrores (email duplicado, salario negativo, depto inexistente...)

**Ejemplos de uso:**
```python
actualizar_empleado(1, salario=50000)                    # Solo el salario
actualizar_empleado(2, nombre='Luis M. Pérez', salario=55000)  # Nombre y salario
actualizar_empleado(999, salario=30000)                  # Devuelve False (no existe)
```

</div>

<div style="background: #FFF8E7; border-left: 6px solid #E8A317; padding: 15px; border-radius: 6px; margin: 10px 0;">

**💡 Hints:**
- `**kwargs` recibe los argumentos como un diccionario: `{'salario': 50000, 'nombre': 'Luis'}`.
- Podéis construir la parte `SET` del UPDATE así: `SET campo1 = ?, campo2 = ?`.
- `cursor.rowcount` os dice cuántas filas se modificaron (0 si el id no existía).
- ⚠️ Cuidado: si cambiáis el `email` a uno que ya existe, saltará un `IntegrityError`.

**💡 Hint avanzado (si os bloqueáis construyendo el SQL dinámico):**
```python
campos = ', '.join(f'{k} = ?' for k in kwargs.keys())
valores = list(kwargs.values())
sql = f"UPDATE empleados SET {campos} WHERE id = ?"
valores.append(id)
conn.execute(query, valores)
```

</div>

In [174]:
# ===========================
# RETO 3: Escribid aquí actualizar_empleado()
# ===========================

def actualizar_empleado(id, **kwargs):
    """
    Actualiza uno o varios campos de un empleado.
    Devuelve True si se actualizó, False si el id no existe.
    """
    if not kwargs:
        return False # No hay campos para actualizar
    campos_a_actualizar = []
    valores = []
    for campo, valor in kwargs.items():
        campos_a_actualizar.append(f"{campo} = ?")
        valores.append(valor)
    # Construir la parte SET de la consulta SQL
    set_clause = ", ".join(campos_a_actualizar)
    sql_query = f"UPDATE empleados SET {set_clause} WHERE id = ?"
    valores.append(id)
    try:
        cursor = conn.execute(sql_query, tuple(valores))
        conn.commit()
        return cursor.rowcount > 0
    except sqlite3.IntegrityError as e:
        conn.rollback()
        print(f"Error de integridad al actualizar empleado: {e}")
        return False
    except Exception as e:
        conn.rollback()
        print(f"Error inesperado al actualizar empleado: {e}")
        return False
    pass

In [175]:
# ===========================
# Probad vuestra función aquí
# ===========================

# Actualizar solo el salario de Ana (id=1)
print(actualizar_empleado(1, salario=50000))

# Verificar
print(buscar_empleado(1))

# Intentar actualizar un empleado que no existe
print(actualizar_empleado(999, salario=30000))

True
{'id': 1, 'nombre': 'Ana García', 'email': 'ana@technova.com', 'salario': 50000.0, 'departamento_id': 1}
False


---

<div style="background: #F5F0FA; border-left: 6px solid #4B2A82; padding: 20px; border-radius: 6px; margin: 20px 0;">

### 🎯 Reto 4 — DELETE

**🔄 Rotación de roles**

Cread una función `borrar_empleado(id)` que:

1. Primero **verifique** si el empleado existe (reutilizad `buscar_empleado()`).
2. Si no existe, devuelva un mensaje informativo.
3. Si existe, lo borre y devuelva un mensaje de confirmación con el nombre del empleado borrado.
4. Maneje errores con `try/except`.

**Probad con:**
- ✅ Borrar un empleado que existe (por ejemplo, el que creasteis en el Reto 1).
- ❌ Intentar borrar un empleado que no existe (id=999).
- 🤔 ¿Qué pasa con sus asignaciones en la tabla `asignaciones`? (Recordad: la BD tiene `ON DELETE CASCADE` en esa tabla).

</div>

<div style="background: #FFF8E7; border-left: 6px solid #E8A317; padding: 15px; border-radius: 6px; margin: 10px 0;">

**💡 Hints:**
- El SQL es simple: `DELETE FROM empleados WHERE id = ?`
- `cursor.rowcount` os dice si se borró algo (1) o no (0).
- Antes de borrar, guardad el nombre del empleado para mostrarlo en el mensaje de confirmación.
- Para ver el efecto del CASCADE, comprobad las asignaciones antes y después: `sql("SELECT * FROM asignaciones WHERE empleado_id = ?", (id,))`

</div>

In [176]:
# ===========================
# RETO 4: Escribid aquí borrar_empleado()
# ===========================

def borrar_empleado(id):
    """
    Borra un empleado por id.
    Devuelve un mensaje informativo (éxito o no encontrado).
    """
    empleado_a_borrar = buscar_empleado(id)
    if empleado_a_borrar is None:
        return f"Empleado con id={id} no encontrado. No se pudo borrar."

    try:
        cursor = conn.execute("DELETE FROM empleados WHERE id = ?", (id,))
        conn.commit()
        if cursor.rowcount > 0:
            return f"Empleado '{empleado_a_borrar['nombre']}' (id={id}) borrado correctamente."
        else:
            conn.rollback()
            return f"No se pudo borrar el empleado con id={id}."
    except Exception as e:
        conn.rollback()
        return f"Error al borrar empleado con id={id}: {e}"

In [177]:
# ===========================
# Probad vuestra función aquí
# ===========================

# Borrar un empleado que existe
print(borrar_empleado(6))  # El que creasteis en el Reto 1

# Intentar borrar uno que no existe
print(borrar_empleado(999))

# Verificar el estado final
sql("SELECT * FROM empleados")

Empleado 'Carlos Vega' (id=6) borrado correctamente.
Empleado con id=999 no encontrado. No se pudo borrar.


,id,nombre,email,salario,departamento_id
0,1,Ana García,ana@technova.com,50000.0,1
1,2,Luis Pérez,luis@technova.com,52000.0,1
2,3,María López,maria@technova.com,48000.0,2
3,4,Pablo Ruiz,pablo@technova.com,55000.0,3
4,5,Sara Díaz,sara@technova.com,60000.0,3


---

## 🔒 Cierre — No olvidéis cerrar la conexión

Cuando terminéis de trabajar, ejecutad esta celda para cerrar la conexión limpiamente.

In [178]:
# Cerrar la conexión
conn.close()
print('🔒 Conexión cerrada')
print('💾 El fichero technova.db sigue en el panel de archivos')

🔒 Conexión cerrada
💾 El fichero technova.db sigue en el panel de archivos


---

## 🔄 Restaurar la BD al estado original

Si necesitas volver a ejecutar las pruebas desde cero, esta celda restaura los datos originales.

In [179]:
# === RESTAURAR DATOS ORIGINALES ===
# Útil si quieres volver a ejecutar todo el notebook desde cero

try:
    conn.rollback()
except:
    pass

try:
    conn.close()
except:
    pass

import os
if os.path.exists('technova.db'):
    os.remove('technova.db')
    print('🗑️ BD anterior eliminada')

# Reconectar (hay que volver a subir el .db original a Colab)
print('⚠️ Vuelve a subir el fichero technova.db original y re-ejecuta desde el Setup')

🗑️ BD anterior eliminada
⚠️ Vuelve a subir el fichero technova.db original y re-ejecuta desde el Setup


---

## 📚 Recursos y documentación CRUD con SQLite

Material de consulta para profundizar en el CRUD con Python y SQLite:

| # | Recurso | Descripción |
|---|---|---|
| 1 | [📘 Documentación oficial de Python — sqlite3](https://docs.python.org/3/library/sqlite3.html) | La referencia completa del módulo `sqlite3`. Incluye tutorial paso a paso, ejemplos de `execute()`, `fetchone()`, `fetchall()`, `commit()`, `row_factory` y manejo de transacciones. **Es la fuente de verdad.** |
| 2 | [🎓 SQLite Tutorial — Python SQLite](https://www.sqlitetutorial.net/sqlite-python/) | Tutorial estructurado por operaciones: conectar, crear tablas, insertar, actualizar, borrar y consultar. Cada sección con ejemplos claros y progresivos. Ideal para seguir paso a paso. |
| 3 | [🌊 DigitalOcean — How To Use sqlite3 in Python 3](https://www.digitalocean.com/community/tutorials/how-to-use-the-sqlite3-module-in-python-3) | Guía práctica completa que cubre CREATE, INSERT, UPDATE y DELETE con un ejemplo de acuario. Explica muy bien los placeholders `?` y la prevención de SQL Injection. |
| 4 | [🐍 Real Python — sqlite3 Module](https://realpython.com/ref/stdlib/sqlite3/) | Referencia con ejemplos prácticos del módulo `sqlite3`. Incluye patrones como `with` para gestión de conexiones y buenas prácticas profesionales. |
| 5 | [🟢 W3Schools — Python SQLite](https://www.w3schools.com/python/ref_module_sqlite3.asp) | Referencia rápida e interactiva con botón "Try it Yourself" para probar el código directamente en el navegador. Muy útil para consultas puntuales. |




---

## 🧠 Recordatorio: operaciones CRUD en SQL

```sql
-- CREATE (Insertar datos)
INSERT INTO usuarios (nombre, edad) VALUES ('Ana', 25);

-- READ (Leer datos)
SELECT * FROM usuarios;

-- UPDATE (Actualizar datos)
UPDATE usuarios SET edad = 26 WHERE nombre = 'Ana';

-- DELETE (Eliminar datos)
DELETE FROM usuarios WHERE nombre = 'Ana';
